# 05 — Independent held-out end-to-end evaluation

Inputs: a trained Generation Agent PEFT checkpoint, trained System-1 classifier/temperature, held-out questions and SQLite databases, plus an optional Grounding model. Generation receives question and retrieved schema only. Gold SQL is read for scoring after candidate selection. Report EX, executable SQL rate, latency, VRAM and ECE; no target metric is prefilled.


In [ ]:
import os
import sys
from pathlib import Path

# Attach a repository snapshot as a Kaggle Dataset, or set KAGGLE_REPO_DIR.
_candidates = [Path(os.environ.get("KAGGLE_REPO_DIR", ".")), Path.cwd(), Path.cwd().parent]
_candidates += list(Path("/kaggle/input").glob("*/Optimize-Mars-Sl")) if Path("/kaggle/input").exists() else []
REPO_DIR = next((p.resolve() for p in _candidates if (p / "src").is_dir()), None)
if REPO_DIR is None:
    raise FileNotFoundError("Set KAGGLE_REPO_DIR to the attached repository directory containing src/")
sys.path.insert(0, str(REPO_DIR))
from src.training.kaggle import KagglePaths, install_dependencies, environment_report
install_dependencies(REPO_DIR)  # Only runs pip when INSTALL_DEPS=1.
paths = KagglePaths.from_env()
SMOKE = paths.smoke
print({"mode": "smoke (NOT a benchmark)" if SMOKE else "train", "data": str(paths.data_dir), "device": paths.device})


In [ ]:
PHASE = 'evaluation'
import json
OUTPUT_DIR = paths.output(PHASE)
environment = environment_report(OUTPUT_DIR)
MAX_STEPS = int(os.environ.get("MAX_STEPS", "100"))
MAX_LENGTH = int(os.environ.get("MAX_LENGTH", "512"))
SEED = int(os.environ.get("SEED", "42"))
SAVE_STEPS = int(os.environ.get("SAVE_STEPS", "25"))
RESUME = os.environ.get("RESUME_FROM_CHECKPOINT") or None  # "latest" or attached checkpoint path
print({"output": OUTPUT_DIR, "resume_from_checkpoint": RESUME, "gpus": environment["gpus"]})


Set `GENERATOR_CHECKPOINT`, `DECISION_CHECKPOINT`, `GROUNDING_CHECKPOINT`, `EVAL_DATA`, `DB_DIR`. `GRAMMAR_MODE=posthoc` labels AST/grammar rejection honestly; `xgrammar` requires the optional dependency and enables constrained decoding for a documented SQL subset. Missing checkpoints fail in real mode. No notebook-local imitation validators are used.

For a leakage audit set `TRAINING_DATA_PATHS` to the SFT, GRPO and System-1 training JSON/JSONL manifests separated by `os.pathsep` (`:` on Kaggle). Any repeated `(db_id, question)` causes evaluation to fail. An omitted manifest is recorded as `training_overlap_checked=false`; verify official held-out split provenance separately. Set `RETRIEVAL_TOP_K`, `MAX_PROMPT_LENGTH`, `MAX_NEW_TOKENS`, `SCORER_DEVICE` and `GROUNDING_DEVICE` to control memory. `posthoc` never claims constrained decoding; the optional XGrammar subset excludes CTEs and window functions.


In [ ]:
from src.evaluation.evaluate_end_to_end import EvaluationHarness
EVAL_DATA = paths.input("EVAL_DATA", "mini_bird.jsonl" if SMOKE else "test.jsonl")
DB_DIR = paths.input("DB_DIR", "." if SMOKE else "databases")
harness = EvaluationHarness(db_dir=DB_DIR,
    num_candidates=int(os.environ.get("NUM_CANDIDATES", "4")),
    model_path=os.environ.get("GENERATOR_CHECKPOINT"),
    decision_model_path=os.environ.get("DECISION_CHECKPOINT"),
    grounding_model_path=os.environ.get("GROUNDING_CHECKPOINT"),
    device=paths.device, dry_run=SMOKE, seed=SEED,
    scorer_device=os.environ.get("SCORER_DEVICE", "cpu"),
    grounding_device=os.environ.get("GROUNDING_DEVICE", "cpu"),
    retrieval_top_k=int(os.environ.get("RETRIEVAL_TOP_K", "25")),
    max_prompt_length=int(os.environ.get("MAX_PROMPT_LENGTH", "2048")),
    max_new_tokens=int(os.environ.get("MAX_NEW_TOKENS", "256")),
    training_data_paths=[p for p in os.environ.get("TRAINING_DATA_PATHS", "").split(os.pathsep) if p],
    constraint_mode=os.environ.get("GRAMMAR_MODE", "posthoc"),
    output_log_dir=str(Path(OUTPUT_DIR) / "logs"), output_report_dir=OUTPUT_DIR)
summary = harness.evaluate_dataset(EVAL_DATA)
print(json.dumps(summary, indent=2))


## Artifact handoff and validation

The last cell returns artifact paths. Download `/kaggle/working/optimize-mars` after a run or save a notebook version with outputs, then attach those outputs to the downstream notebook as a Kaggle Dataset. Runtime storage is ephemeral; a local checkpoint alone does not survive session deletion. Export before the session ends. Set `RESUME_FROM_CHECKPOINT` to an attached complete checkpoint, or `latest` for the current output directory. No checkpoints are automatically deleted.

Use `PIPELINE_MODE=smoke` for offline wiring checks; this mode does not train pretrained models or measure model quality. Default `train` requires real datasets, dependencies, GPU and the input artifacts listed above. Settings target **one 16 GB T4**, sequential rollouts and short sequences; T4 x2 does not provide a shared 32 GB memory pool. Reduce `MAX_LENGTH`, rollout length or accumulation if needed. No full GPU run or Kaggle benchmark is claimed by this notebook. See `KAGGLE.md` for data contracts, independent account handoffs and ablations.
